# Example: static and dynamic topic modeling with BERTopic on a synthetic dataset

This notebook runs the pipeline of the repository end to end on a small synthetic dataset, so that it can be tried without access to the copyrighted newspaper corpus.

**About the data.** `dummy_newspaper_dataset.csv` contains 2,880 invented newspaper articles dated 1955 to 2018. They are generated from sentence templates by `generate_dummy_dataset.py`; places, people and events are fictional. Each article belongs to one of nine themes, and the share of each theme changes over the years. Because the texts are template-based, the themes are much easier to separate than topics in real newspapers. The example shows how the workflow runs and what its outputs look like. It says nothing about the quality of the method on real data.

In [1]:
import re
import warnings

import nltk
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

warnings.filterwarnings("ignore")
for resource in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

## 1. Load the data

In [2]:
df = pd.read_csv("dummy_newspaper_dataset.csv", sep=";")
print(len(df), "articles,", df["year"].min(), "to", df["year"].max())
df[["uid", "date", "title", "content"]].head()

2880 articles, 1955 to 2018


,uid,date,title,content
0,DUMMY-1955-00000,1955-09-24,Coalition talks begin after the election,Members of the council approved the new budget...
1,DUMMY-1955-00001,1955-02-07,Bus service extended to Eastbrook,Commuters welcomed the faster connection and t...
2,DUMMY-1955-00002,1955-03-17,Milk prices worry farmers in Marlow Valley,Cattle and sheep were sold at the weekly lives...
3,DUMMY-1955-00003,1955-09-28,Cattle market draws crowds to Riverton,"Fertiliser costs rose sharply, reducing the in..."
4,DUMMY-1955-00004,1955-09-04,Storm damage across the region,Flood water covered streets and cellars in the...


## 2. Preprocess

The same steps as for the real corpus: lowercase, remove punctuation and stopwords, drop words of one or two characters, and lemmatise.

In [3]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    return " ".join(lemmatizer.lemmatize(word) for word in text.split()
                    if len(word) > 2 and word not in stop_words)

df["content_clean"] = df["content"].apply(preprocess)
df["content_clean"].iloc[0]

'member council approved new budget long debate coalition talk two largest party began week reporter newspaper present scene candidate thornbury elected mayor marlow valley clear majority election commission confirmed final result wednesday campaign focused tax pension housing governing party lost seat opposition gained vote ballot paper counted night town hall opposition leader called vote confidence parliament'

## 3. Compute the document embeddings

Each document is embedded as a whole with `Alibaba-NLP/gte-base-en-v1.5`, the embedding model used in the paper.

In [4]:
embedding_model = SentenceTransformer("Alibaba-NLP/gte-base-en-v1.5", trust_remote_code=True)

contents = df["content_clean"].tolist()
years = df["year"].tolist()
embeddings = embedding_model.encode(contents, show_progress_bar=False)
embeddings.shape

(2880, 768)

## 4. Static topic model

BERTopic is fitted once on all documents. UMAP reduces the embeddings, HDBSCAN clusters them, and c-TF-IDF extracts the words of each topic. KeyBERT-inspired and MMR representations are added, as in the paper.

`min_cluster_size` is set to 40 here because the synthetic dataset is small and repetitive. For the real corpus the paper uses the BERTopic defaults and a tuned configuration.

In [5]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=40, metric="euclidean", cluster_selection_method="eom", prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english")
representation_model = {
    "KeyBERT": KeyBERTInspired(),
    "MMR": MaximalMarginalRelevance(diversity=0.3),
}

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    top_n_words=10,
)
topics, probs = topic_model.fit_transform(contents, embeddings)

In [6]:
topic_info = topic_model.get_topic_info()
topic_info[["Topic", "Count", "Representation", "KeyBERT"]]

,Topic,Count,Representation,KeyBERT
0,0,435,"[farmer, farm, agricultural, harvest, cattle, ...","[agricultural, farmer, farm, harvesting, harve..."
1,1,414,"[goal, supporter, league, club, season, match,...","[football, derby, championship, league, goalke..."
2,2,405,"[train, new, railway, traffic, commuter, trans...","[transport, motorway, traffic, railway, train,..."
3,3,315,"[patient, clinic, doctor, health, time, waitin...","[health, clinic, medicine, hospital, epidemic,..."
4,4,307,"[computer, online, software, year, website, fo...","[digital, computer, online, developed, develop..."
5,5,306,"[school, received, library, student, donation,...","[education, school, primary, classroom, librar..."
6,6,273,"[party, election, vote, opposition, parliament...","[election, news, newspaper, elected, announced..."
7,7,272,"[electricity, power, supply, fuel, station, pr...","[energy, electricity, power, fuel, petrol, new..."
8,8,153,"[water, flood, river, heavy, rain, home, road,...","[flood, flooded, rain, evacuated, riverton, st..."


Since the dataset is synthetic, the theme of every article is known. The table below compares the topics found by BERTopic with those themes.

In [7]:
pd.crosstab(df["theme"], pd.Series(topics, name="topic"))

topic,0,1,2,3,4,5,6,7,8
theme,,,,,,,,,
agriculture,435,0,0,0,0,0,0,0,0
computing,0,0,0,0,307,0,0,0,0
education,0,0,0,0,0,306,0,0,0
elections,0,0,0,0,0,0,273,0,0
energy,0,0,0,0,0,0,0,272,0
floods,0,0,0,0,0,0,0,0,153
football,0,414,0,0,0,0,0,0,0
health,0,0,0,315,0,0,0,0,0
transport,0,0,405,0,0,0,0,0,0


## 5. Dynamic topic modeling: topics over time

The fitted model is not retrained. The documents are grouped by publication year, and the frequency and the words of each topic are computed per year.

In [8]:
topics_over_time = topic_model.topics_over_time(contents, years)
topics_over_time.head(10)

,Topic,Words,Frequency,Timestamp
0,0,"farm, farmer, cattle, agricultural, harvest",8,1955
1,1,"supporter, goal, match, league, club",4,1955
2,2,"train, ticket, new, transport, railway",9,1955
3,3,"patient, clinic, health, doctor, treatment",5,1955
4,5,"school, student, received, record, number",2,1955
5,6,"election, party, vote, opposition, parliament",13,1955
6,7,"power, supply, electricity, fuel, energy",2,1955
7,8,"water, flood, river, rain, barrier",2,1955
8,0,"farmer, farm, agricultural, cattle, harvest",11,1956
9,1,"season, supporter, team, league, club",7,1956


In [9]:
# Years in which each topic is most frequent
peaks = (topics_over_time.sort_values("Frequency", ascending=False)
         .groupby("Topic").head(3)
         .sort_values(["Topic", "Frequency"], ascending=[True, False]))
peaks = peaks.merge(topic_info[["Topic", "Name"]], on="Topic")
peaks[["Topic", "Name", "Timestamp", "Frequency"]]

,Topic,Name,Timestamp,Frequency
0,0,0_farmer_farm_agricultural_harvest,1963,14
1,0,0_farmer_farm_agricultural_harvest,1969,13
2,0,0_farmer_farm_agricultural_harvest,1971,12
3,1,1_goal_supporter_league_club,1976,13
4,1,1_goal_supporter_league_club,2012,12
5,1,1_goal_supporter_league_club,1987,12
6,2,2_train_new_railway_traffic,1965,11
7,2,2_train_new_railway_traffic,1962,10
8,2,2_train_new_railway_traffic,1960,10
9,3,3_patient_clinic_doctor_health,1969,16


## 6. Visualisations

The interactive Plotly figures are saved as HTML files in `docs/figures/` and shown on the project page.

In [10]:
def save_figure(fig, name):
    fig.update_layout(width=None, autosize=True)
    fig.write_html(f"../docs/figures/{name}.html", include_plotlyjs="cdn")

save_figure(topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10), "topics_over_time")
save_figure(topic_model.visualize_barchart(top_n_topics=10, n_words=8), "topic_barchart")
save_figure(topic_model.visualize_hierarchy(), "topic_hierarchy")

topic_info[["Topic", "Count", "Name", "Representation"]].to_csv("../docs/figures/topic_info.csv", index=False)

## 7. Following particular events

A sudden rise of one topic in the topics-over-time data points to an event that was widely reported. The synthetic dataset contains three kinds of invented events: fuel shortages, floods and an influenza wave. The cells below pick out the topics they belong to, compare each topic's share of the articles in its peak years with its usual share, and list the words that describe the topic in those years.

In [11]:
def find_topic(keyword):
    """Return the topic whose top words contain the keyword."""
    for topic, words in topic_model.get_topics().items():
        if topic != -1 and keyword in [word for word, _ in words]:
            return topic

event_topics = {
    "Fuel shortages": find_topic("fuel"),
    "Floods": find_topic("flood"),
    "Influenza wave": find_topic("influenza"),
}
event_topics

{'Fuel shortages': 7, 'Floods': 8, 'Influenza wave': 3}

In [12]:
articles_per_year = df.groupby("year").size()

rows = []
for event, topic in event_topics.items():
    series = topics_over_time[topics_over_time["Topic"] == topic].set_index("Timestamp")
    share = (series["Frequency"] / articles_per_year).fillna(0) * 100
    usual = share.median()
    for year in share.sort_values(ascending=False).head(4).index:
        if share[year] >= 2 * usual:
            rows.append({"Event": event, "Topic": topic, "Year": year,
                         "Articles": int(series.loc[year, "Frequency"]),
                         "Share of the year (%)": round(share[year], 1),
                         "Usual share (%)": round(usual, 1),
                         "Words in that year": series.loc[year, "Words"]})
event_table = pd.DataFrame(rows).sort_values(["Event", "Year"]).reset_index(drop=True)
event_table

,Event,Topic,Year,Articles,Share of the year (%),Usual share (%),Words in that year
0,Floods,8,1962,14,31.1,4.4,"flood, water, rain, river, heavy"
1,Floods,8,1978,11,24.4,4.4,"flood, water, river, heavy, rain"
2,Floods,8,1993,15,33.3,4.4,"water, flood, rain, river, heavy"
3,Floods,8,2002,15,33.3,4.4,"water, flood, heavy, rain, river"
4,Fuel shortages,7,1973,15,33.3,8.9,"electricity, power, supply, fuel, station"
5,Fuel shortages,7,1974,17,37.8,8.9,"power, electricity, supply, fuel, station"
6,Fuel shortages,7,2008,14,31.1,8.9,"power, electricity, station, supply, fuel"
7,Influenza wave,3,1968,16,35.6,11.1,"patient, clinic, doctor, health, ray"
8,Influenza wave,3,1969,16,35.6,11.1,"patient, clinic, doctor, health, vaccination"
9,Influenza wave,3,1981,11,24.4,11.1,"patient, clinic, doctor, health, time"


In [13]:
# One figure per event
for event, topic in event_topics.items():
    fig = topic_model.visualize_topics_over_time(topics_over_time, topics=[topic], title=f"<b>{event}</b>")
    fig.update_layout(showlegend=False)
    previous_year = None
    for _, row in event_table[event_table["Event"] == event].sort_values("Year").iterrows():
        # raise the label of a year that directly follows another labelled year
        offset = -48 if previous_year is not None and row["Year"] - previous_year <= 2 else -28
        fig.add_annotation(x=row["Year"], y=row["Articles"], text=str(row["Year"]),
                           showarrow=True, arrowhead=2, ax=0, ay=offset, font=dict(size=12))
        previous_year = row["Year"]
    save_figure(fig, "event_" + event.lower().replace(" ", "_"))

event_table.to_csv("../docs/figures/event_table.csv", index=False)

All years of the invented events are found. The rule also flags 1981 and 1999 for the health topic, where no event was built in: with only 45 articles per year, chance alone can double a topic's share. A peak is a reason to read the documents behind it, not proof of an event.